In [1]:
import cupy as cp

Class implementation

In [ ]:
class BatchNorm2D:
    """Batch normalization for 4D inputs (N, C, H, W), CuPy-only.

    Normalizes each channel over the NxHxW slice. Running statistics follow PyTorch: 
    ``running = (1 - momentum) * running + momentum * batch_stat``.
    Batch variance uses the biased estimator (ddof=0), matching ``torch.nn.BatchNorm2d``.
    """

    def __init__(
        self,
        num_features: int,
        eps: float = 1e-5,
        momentum: float = 0.1,
        affine: bool = True,
        dtype=cp.float64,
    ):
        self.num_features = int(num_features)
        self.eps = float(eps)
        self.momentum = float(momentum)
        self.affine = bool(affine)
        self.dtype = dtype
        self.training = True

        self.running_mean = cp.zeros(self.num_features, dtype=dtype)
        self.running_var = cp.ones(self.num_features, dtype=dtype)

        if self.affine:
            self.gamma = cp.ones(self.num_features, dtype=dtype)
            self.beta = cp.zeros(self.num_features, dtype=dtype)
        else:
            self.gamma = None
            self.beta = None

        self.dgamma = None
        self.dbeta = None
        self._x = None
        self._x_hat = None
        self._std = None
        self._shape = None

    def train(self, mode: bool = True) -> None:
        self.training = bool(mode)

    def eval(self) -> None:
        self.training = False

    def forward(self, x: cp.ndarray) -> cp.ndarray:
        assert x.ndim == 4, f"Expected (N, C, H, W), got shape {x.shape}"
        _, c, _, _ = x.shape
        assert c == self.num_features, (
            f"Input has {c} channels, expected {self.num_features}"
        )

        self._shape = x.shape
        reduce_axes = (0, 2, 3)

        if self.training:
            mean = x.mean(axis=reduce_axes)
            var = x.var(axis=reduce_axes, ddof=0)
            self.running_mean = (1.0 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1.0 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean
            var = self.running_var

        mean_bc = mean[None, :, None, None]
        var_bc = var[None, :, None, None]
        std = cp.sqrt(var_bc + self.eps)
        x_hat = (x - mean_bc) / std

        self._x = x
        self._x_hat = x_hat
        self._std = std

        if self.affine:
            out = self.gamma[None, :, None, None] * x_hat + self.beta[None, :, None, None]
        else:
            out = x_hat
        return out

    def backward(self, grad_output: cp.ndarray) -> cp.ndarray:
        assert self._x is not None, "Run forward() before backward()"
        assert grad_output.shape == self._shape

        reduce_axes = (0, 2, 3)

        if self.affine:
            gamma_bc = self.gamma[None, :, None, None]
            dx_hat = grad_output * gamma_bc
            self.dgamma = (grad_output * self._x_hat).sum(axis=reduce_axes)
            self.dbeta = grad_output.sum(axis=reduce_axes)
        else:
            dx_hat = grad_output
            self.dgamma = None
            self.dbeta = None

        x_hat = self._x_hat
        std = self._std
        mean_dx = dx_hat.mean(axis=reduce_axes, keepdims=True)
        mean_dx_xhat = (dx_hat * x_hat).mean(axis=reduce_axes, keepdims=True)
        dx = (dx_hat - mean_dx - x_hat * mean_dx_xhat) / std
        return dx

    def __call__(self, x: cp.ndarray) -> cp.ndarray:
        return self.forward(x)

    def __repr__(self) -> str:
        return (
            f"BatchNorm2D(num_features={self.num_features}, eps={self.eps}, "
            f"momentum={self.momentum}, affine={self.affine})"
        )

Tests

In [3]:
cp.random.seed(42)


def _bn_loss(x: cp.ndarray, grad_out: cp.ndarray, gamma: cp.ndarray, beta: cp.ndarray, C: int) -> float:
    """Scalar L = sum(forward(x) * grad_out) with a fresh layer (momentum=0 so buffers do not drift)."""
    bn = BatchNorm2D(C, eps=1e-5, momentum=0.0, affine=True, dtype=cp.float64)
    bn.gamma[:] = gamma
    bn.beta[:] = beta
    bn.train(True)
    y = bn.forward(x)
    return float(cp.sum(y * grad_out))


# --- Forward: match explicit batch-norm formula ---
N, C, H, W = 2, 4, 3, 3
x = cp.random.randn(N, C, H, W).astype(cp.float64)
bn = BatchNorm2D(C, eps=1e-5, momentum=0.1, affine=True, dtype=cp.float64)
bn.gamma = cp.random.uniform(0.5, 1.5, (C,)).astype(cp.float64)
bn.beta = cp.random.randn(C).astype(cp.float64)
bn.train(True)
y = bn.forward(x)

mean = x.mean(axis=(0, 2, 3))
var = x.var(axis=(0, 2, 3), ddof=0)
mean_bc = mean[None, :, None, None]
std = cp.sqrt(var[None, :, None, None] + bn.eps)
x_hat = (x - mean_bc) / std
expected_y = bn.gamma[None, :, None, None] * x_hat + bn.beta[None, :, None, None]
assert cp.allclose(y, expected_y, rtol=1e-12, atol=1e-12), "Forward differs from explicit formula"

# --- Gradients w.r.t. gamma and beta ---
grad_out = cp.random.randn(N, C, H, W).astype(cp.float64)
dx = bn.backward(grad_out)
dgamma_exp = (grad_out * x_hat).sum(axis=(0, 2, 3))
dbeta_exp = grad_out.sum(axis=(0, 2, 3))
assert cp.allclose(bn.dgamma, dgamma_exp, rtol=1e-12, atol=1e-12)
assert cp.allclose(bn.dbeta, dbeta_exp, rtol=1e-12, atol=1e-12)

# --- Numeric check for dx (central differences) ---
gamma0 = cp.random.uniform(0.5, 1.5, (C,)).astype(cp.float64)
beta0 = cp.random.randn(C).astype(cp.float64)
x0 = cp.random.randn(2, C, 2, 2).astype(cp.float64)
g0 = cp.random.randn(2, C, 2, 2).astype(cp.float64)

bn_num = BatchNorm2D(C, eps=1e-5, momentum=0.0, affine=True, dtype=cp.float64)
bn_num.gamma[:] = gamma0
bn_num.beta[:] = beta0
bn_num.train(True)
bn_num.forward(x0)
dx_ana = bn_num.backward(g0)

step = 1e-5
dx_num = cp.zeros_like(x0)
flat = x0.ravel()
for k in range(flat.size):
    xp = x0.copy()
    xm = x0.copy()
    xp.ravel()[k] += step
    xm.ravel()[k] -= step
    dx_num.ravel()[k] = (_bn_loss(xp, g0, gamma0, beta0, C) - _bn_loss(xm, g0, gamma0, beta0, C)) / (2 * step)

rel_err = float(cp.linalg.norm(dx_ana - dx_num) / (cp.linalg.norm(dx_num) + 1e-12))
assert rel_err < 1e-4, f"dx analytic vs numeric rel error too large: {rel_err}"

# --- Eval mode uses running statistics ---
bn_eval = BatchNorm2D(C, eps=1e-5, momentum=0.0, affine=True, dtype=cp.float64)
bn_eval.running_mean = cp.arange(C, dtype=cp.float64)
bn_eval.running_var = cp.arange(C, dtype=cp.float64) + 1.0
bn_eval.eval()
y_eval = bn_eval.forward(x[:, :C, :, :])
expected_eval = (
    bn_eval.gamma[None, :, None, None]
    * (x[:, :C, :, :] - bn_eval.running_mean[None, :, None, None])
    / cp.sqrt(bn_eval.running_var[None, :, None, None] + bn_eval.eps)
    + bn_eval.beta[None, :, None, None]
)
assert cp.allclose(y_eval, expected_eval, rtol=1e-12, atol=1e-12)

print("All tests passed.")

All tests passed.
